# Post Vector Store — Three-Partition System
**No ChromaDB. No external vector DB. Pure numpy + pandas.**

Persists to 3 `.pkl` files on disk:
- `./vector_store/posts_hot.pkl` — posts created in last **72 hours**
- `./vector_store/posts_warm.pkl` — posts created **72 hrs → 5 days** ago
- `./vector_store/posts_cold.pkl` — posts created **5 → 30 days** ago

### How `expires_at` is set at post creation
When a user hits Publish, the backend does one line:
```python
expires_at = created_at + timedelta(days=CATEGORY_EXPIRY_DAYS[category])
```
| Category | Expiry | Lives in |
|---|---|---|
| market_update | 2 days | hot only |
| deal_req | 7 days | hot + warm |
| discussion | 14 days | hot + warm |
| knowledge | 90 days | hot + warm + cold |

### Vector layout (18 dims)
```
[0:10]  commodity one-hot   (rice, wheat, cotton, sugar, soybean, maize, spices, pulses, groundnut, mustard)
[10:13] role multi-hot      (trader, exporter, broker)
[13:16] geo 3D cartesian    (lat/lon → unit sphere x,y,z)
[16:18] qty normalised      (0–1 over 5000 MT scale — zeros for non-deal posts)
```

In [1]:
import math
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta, timezone
from copy import deepcopy

print("numpy", np.__version__, "| pandas", pd.__version__)

numpy 2.4.3 | pandas 3.0.1


## Cell 2 — Constants & lookup tables

In [2]:
# Change CSV_PATH to point to your posts_vector_db.csv
CSV_PATH  = "posts_vector_db.csv"
STORE_DIR = Path("./vector_store")
STORE_DIR.mkdir(exist_ok=True)

# Reference time — "now" for this dataset
CUTOFF    = datetime(2026, 4, 3, 10, 0, 0, tzinfo=timezone.utc)

# Partition boundaries
HOT_HRS   = 72       # posts younger than 72 hrs → hot
WARM_DAYS = 5        # posts 72 hrs–5 days → warm
COLD_DAYS = 30       # posts 5–30 days → cold

# Vector dimensions
COMMODITIES = ["rice","wheat","cotton","sugar","soybean",
               "maize","spices","pulses","groundnut","mustard"]
ROLES       = ["trader","exporter","broker"]

# How long each category lives — set at post-creation time
CATEGORY_EXPIRY_DAYS = {
    "market_update": 2,
    "deal_req":      7,
    "discussion":    14,
    "knowledge":     90,
}

# Which partitions each category is allowed to live in
CATEGORY_PARTITIONS = {
    "market_update": ["hot"],
    "deal_req":      ["hot", "warm"],
    "discussion":    ["hot", "warm"],
    "knowledge":     ["hot", "warm", "cold"],
}

print("Constants loaded.")
print(f"STORE_DIR : {STORE_DIR.resolve()}")
print(f"CUTOFF    : {CUTOFF}")

Constants loaded.
STORE_DIR : C:\Users\Admin\OneDrive\Desktop\Connection Recommend\POST RECOMMED\vector_store
CUTOFF    : 2026-04-03 10:00:00+00:00


## Cell 3 — Partition class
One instance per collection (hot / warm / cold). Holds vectors + metadata in memory, persists to `.pkl`.

In [3]:
class Partition:
    """
    In-memory vector store for one time partition.

    Internally:
        self.ids      : list[str]           — post_ids
        self.matrix   : np.ndarray (N, 18)  — vectors stacked row-wise
        self.metadata : list[dict]          — payload per post
        self._id_index: dict[str, int]      — id → row index for O(1) lookup
    """

    def __init__(self, name: str):
        self.name       = name
        self._path      = STORE_DIR / f"posts_{name}.pkl"
        self.ids        = []
        self.matrix     = np.empty((0, 18), dtype=np.float32)
        self.metadata   = []
        self._id_index  = {}
        self._load()

    # ── Persistence ──────────────────────────────────────────────────────────

    def _load(self):
        if self._path.exists():
            with open(self._path, "rb") as f:
                data = pickle.load(f)
            self.ids       = data["ids"]
            self.matrix    = data["matrix"]
            self.metadata  = data["metadata"]
            self._id_index = {id_: i for i, id_ in enumerate(self.ids)}
            print(f"  Loaded posts_{self.name}: {len(self.ids):,} vectors from disk")

    def save(self):
        with open(self._path, "wb") as f:
            pickle.dump({"ids": self.ids,
                         "matrix": self.matrix,
                         "metadata": self.metadata}, f,
                        protocol=pickle.HIGHEST_PROTOCOL)

    # ── Write ─────────────────────────────────────────────────────────────────

    def upsert(self, post_id: str, vector: list, metadata: dict):
        """Add new post or update existing one in-place."""
        vec = np.array(vector, dtype=np.float32)
        if post_id in self._id_index:
            idx = self._id_index[post_id]
            self.matrix[idx]   = vec
            self.metadata[idx] = metadata
        else:
            idx = len(self.ids)
            self.ids.append(post_id)
            self.matrix = (np.vstack([self.matrix, vec])
                           if self.matrix.shape[0] > 0
                           else vec.reshape(1, -1))
            self.metadata.append(metadata)
            self._id_index[post_id] = idx

    def delete(self, post_ids: list):
        """Remove rows by id. Rebuilds index (used infrequently by expiry job)."""
        remove = set(post_ids)
        keep   = [i for i, id_ in enumerate(self.ids) if id_ not in remove]
        self.ids       = [self.ids[i] for i in keep]
        self.matrix    = (self.matrix[keep] if keep
                          else np.empty((0, 18), dtype=np.float32))
        self.metadata  = [self.metadata[i] for i in keep]
        self._id_index = {id_: i for i, id_ in enumerate(self.ids)}

    # ── Query ─────────────────────────────────────────────────────────────────

    def query(self, user_vector: list, filters: dict,
              top_k: int = 120, seen_ids: set = None) -> list:
        """
        Batched cosine similarity against all rows, then filter.

        Supported filter keys:
            is_active : int   (1 or 0)
            category  : str or list[str]
            commodity : str
        """
        if self.matrix.shape[0] == 0:
            return []

        seen = seen_ids or set()
        N    = len(self.ids)
        mask = np.ones(N, dtype=bool)

        for i, (id_, meta) in enumerate(zip(self.ids, self.metadata)):
            if id_ in seen:
                mask[i] = False; continue
            if "is_active" in filters and meta.get("is_active") != filters["is_active"]:
                mask[i] = False; continue
            if "category" in filters:
                cats = filters["category"]
                if isinstance(cats, str):
                    cats = [cats]
                if meta.get("category") not in cats:
                    mask[i] = False; continue
            if "commodity" in filters and meta.get("commodity") != filters["commodity"]:
                mask[i] = False; continue

        candidate_idx = np.where(mask)[0]
        if len(candidate_idx) == 0:
            return []

        # batched cosine similarity (vectorised — fast on numpy)
        sub    = self.matrix[candidate_idx]
        q      = np.array(user_vector, dtype=np.float32)
        dots   = sub @ q
        norms  = np.linalg.norm(sub, axis=1) * np.linalg.norm(q)
        norms  = np.where(norms == 0, 1e-9, norms)
        scores = dots / norms

        top_k = min(top_k, len(candidate_idx))
        top_i = np.argpartition(scores, -top_k)[-top_k:]
        top_i = top_i[np.argsort(scores[top_i])[::-1]]

        return [{
            "post_id":   self.ids[candidate_idx[i]],
            "score":     round(float(scores[i]), 6),
            "partition": self.name,
            **self.metadata[candidate_idx[i]],
        } for i in top_i]

    # ── Get by filter (for expiry job) ────────────────────────────────────────

    def get_where(self, filters: dict) -> list:
        """
        Returns [(post_id, row_idx, metadata)] for rows matching filters.
        Supports operators: {"$eq": v} and {"$lte": v}
        """
        results = []
        for i, (id_, meta) in enumerate(zip(self.ids, self.metadata)):
            match = True
            for k, v in filters.items():
                actual = meta.get(k)
                if isinstance(v, dict):
                    op, val = list(v.items())[0]
                    if actual is None:
                        match = False; break
                    if op == "$lte" and not (actual <= val):
                        match = False; break
                    if op == "$eq"  and actual != val:
                        match = False; break
                else:
                    if actual != v:
                        match = False; break
            if match:
                results.append((id_, i, meta))
        return results

    @property
    def count(self):
        return len(self.ids)

print("Partition class defined.")

Partition class defined.


## Cell 4 — PostVectorStore class
Manages all 3 partitions. This is the main interface your backend calls.

In [23]:
class PostVectorStore:
    """
    Main interface. Three methods your backend uses:
        store.upsert_post(row)          — called when user publishes a post
        store.query(vec, filters, ...)  — called when user opens feed
        store.run_expiry_job()          — called every hour by cron / Celery
    """

    def __init__(self):
        self.hot    = Partition("hot")
        self.warm   = Partition("warm")
        self.cold   = Partition("cold")
        self._parts = {"hot": self.hot, "warm": self.warm, "cold": self.cold}

    def save_all(self):
        for p in self._parts.values():
            p.save()
        print("Saved all partitions to disk.")

    # ═══════════════════════════════════════════════════════════════════════
    # USER VECTOR
    # Built from the user's own profile at query time (feed load).
    # Represents WHO the user IS and WHAT they trade.
    #
    # Dims:
    #   [0:10]  commodity one-hot  — the commodity this user trades
    #   [10:13] role multi-hot     — what role this user IS (trader/exporter/broker)
    #   [13:16] geo 3D cartesian   — where this user is located
    #   [16:18] qty normalised     — this user's typical trade size (optional)
    # ═══════════════════════════════════════════════════════════════════════

    @staticmethod
    def build_user_vector(commodity: str,
                          user_role: str,
                          latitude: float,
                          longitude: float,
                          qty_min_mt: float = None,
                          qty_max_mt: float = None) -> list:
        """
        Called at feed-load time to build the query vector for a user.

        Args:
            commodity  : user's primary commodity e.g. "rice"
            user_role  : what the user IS — "trader" / "exporter" / "broker"
                         (single value, not pipe-separated — a user has one role)
            latitude   : user's location
            longitude  : user's location
            qty_min_mt : user's typical minimum trade size (optional)
            qty_max_mt : user's typical maximum trade size (optional)

        Example:
            vec = PostVectorStore.build_user_vector(
                commodity  = "rice",
                user_role  = "trader",
                latitude   = 19.0760,
                longitude  = 72.8777,
                qty_min_mt = 50,
                qty_max_mt = 200,
            )
        """
        # [0:10] commodity one-hot
        com = [1.0 if c == commodity else 0.0 for c in COMMODITIES]

        # [10:13] role multi-hot — user has exactly one role
        rol = [1.0 if r == user_role else 0.0 for r in ROLES]

        # [13:16] geo 3D cartesian
        lat_r = math.radians(float(latitude))
        lon_r = math.radians(float(longitude))
        geo   = [
            round(math.cos(lat_r) * math.cos(lon_r), 6),
            round(math.cos(lat_r) * math.sin(lon_r), 6),
            round(math.sin(lat_r), 6),
        ]

        # [16:18] qty — pass user's trade size if known, else zeros
        if (qty_min_mt is not None
                and not (isinstance(qty_min_mt, float) and math.isnan(qty_min_mt))):
            qty = [round(min(float(qty_min_mt) / 5000.0, 1.0), 4),
                   round(min(float(qty_max_mt) / 5000.0, 1.0), 4)]
        else:
            qty = [0.0, 0.0]

        return com + rol + geo + qty   # 18 dims

    # ═══════════════════════════════════════════════════════════════════════
    # POST VECTOR
    # Built from the post's fields at write time (post publish).
    # Represents WHO the post TARGETS and WHAT it is about.
    #
    # Dims:
    #   [0:10]  commodity one-hot  — the commodity this post is about
    #   [10:13] role multi-hot     — which roles this post TARGETS (one or many)
    #   [13:16] geo 3D cartesian   — where this post is located / relevant
    #   [16:18] qty normalised     — deal size (deal_req only, else 0.0)
    # ═══════════════════════════════════════════════════════════════════════

    @staticmethod
    def build_post_vector(commodity: str,
                          target_roles_str: str,
                          latitude: float,
                          longitude: float,
                          category: str,
                          qty_min_mt: float = None,
                          qty_max_mt: float = None) -> list:
        """
        Called at post-publish time to build the vector stored in the DB.

        Args:
            commodity        : commodity this post is about e.g. "rice"
            target_roles_str : who this post targets — pipe-separated
                               e.g. "trader" or "trader|broker"
                               (posts can target multiple roles)
            latitude         : post's location
            longitude        : post's location
            category         : post category — used to decide if qty dims are
                               populated (only deal_req gets qty > 0)
            qty_min_mt       : deal size min (deal_req only)
            qty_max_mt       : deal size max (deal_req only)

        Example:
            vec = PostVectorStore.build_post_vector(
                commodity        = "rice",
                target_roles_str = "trader|broker",
                latitude         = 19.0760,
                longitude        = 72.8777,
                category         = "deal_req",
                qty_min_mt       = 100,
                qty_max_mt       = 400,
            )
        """
        # [0:10] commodity one-hot
        com = [1.0 if c == commodity else 0.0 for c in COMMODITIES]

        # [10:13] role multi-hot — post can target multiple roles
        roles_set = set(str(target_roles_str).split("|"))
        rol = [1.0 if r in roles_set else 0.0 for r in ROLES]

        # [13:16] geo 3D cartesian
        lat_r = math.radians(float(latitude))
        lon_r = math.radians(float(longitude))
        geo   = [
            round(math.cos(lat_r) * math.cos(lon_r), 6),
            round(math.cos(lat_r) * math.sin(lon_r), 6),
            round(math.sin(lat_r), 6),
        ]

        # [16:18] qty — only populated for deal_req, zeros for all other categories
        if (category == "deal_req"
                and qty_min_mt is not None
                and not (isinstance(qty_min_mt, float) and math.isnan(qty_min_mt))):
            qty = [round(min(float(qty_min_mt) / 5000.0, 1.0), 4),
                   round(min(float(qty_max_mt) / 5000.0, 1.0), 4)]
        else:
            qty = [0.0, 0.0]

        return com + rol + geo + qty   # 18 dims

    # ── Partition assignment ───────────────────────────────────────────────

    @staticmethod
    def assign_partition(created_at_str) -> str:
        created_at = datetime.fromisoformat(str(created_at_str))
        if created_at.tzinfo is None:
            created_at = created_at.replace(tzinfo=timezone.utc)
        age_hours = (CUTOFF - created_at).total_seconds() / 3600
        if   age_hours <= HOT_HRS:        return "hot"
        elif age_hours <= WARM_DAYS * 24: return "warm"
        else:                             return "cold"

    # ── Write (called at post-publish time) ───────────────────────────────

    def upsert_post(self, row: dict) -> str:
        partition = self.assign_partition(row["created_at"])
        category  = row["category"]

        if partition not in CATEGORY_PARTITIONS.get(category, []):
            return None

        # Uses build_post_vector — category decides qty dims
        vector = self.build_post_vector(
            commodity        = row["commodity"],
            target_roles_str = row["target_roles"],
            latitude         = row["latitude"],
            longitude        = row["longitude"],
            category         = category,
            qty_min_mt       = row.get("qty_min_mt"),
            qty_max_mt       = row.get("qty_max_mt"),
        )

        metadata = {
            "category":       category,
            "commodity":      str(row["commodity"]),
            "target_roles":   str(row["target_roles"]),
            "location_state": str(row.get("location_state", "")),
            "location_city":  str(row.get("location_city", "")),
            "latitude":       float(row["latitude"]),
            "longitude":      float(row["longitude"]),
            "qty_min_mt":     float(row["qty_min_mt"]) if pd.notna(row.get("qty_min_mt")) else 0.0,
            "qty_max_mt":     float(row["qty_max_mt"]) if pd.notna(row.get("qty_max_mt")) else 0.0,
            "price_inr":      float(row["price_per_unit_inr"]) if pd.notna(row.get("price_per_unit_inr")) else 0.0,
            "created_at":     str(row["created_at"]),
            "expires_at":     str(row["expires_at"]),
            "is_active":      1,
            "partition":      partition,
        }

        self._parts[partition].upsert(str(row["post_id"]), vector, metadata)
        return partition

    # ── Query (called when user opens feed) ───────────────────────────────

    def query(self, user_vector: list, filters: dict,
              top_k: int = 120, seen_ids: set = None) -> list:
        seen     = set(seen_ids or [])
        all_hits = []
        for pname in ["hot", "warm", "cold"]:
            hits = self._parts[pname].query(user_vector, filters,
                                            top_k=top_k, seen_ids=seen)
            all_hits.extend(hits)
            seen.update(h["post_id"] for h in hits)
        all_hits.sort(key=lambda x: x["score"], reverse=True)
        return all_hits[:top_k]

    # ── Expiry job (run every hour via cron / Celery beat) ────────────────

    def run_expiry_job(self, reference_time=None) -> dict:
        now         = reference_time or datetime.now(timezone.utc)
        now_iso     = now.isoformat()
        hot_cutoff  = (now - timedelta(hours=HOT_HRS)).isoformat()
        warm_cutoff = (now - timedelta(days=WARM_DAYS)).isoformat()
        cold_cutoff = (now - timedelta(days=COLD_DAYS)).isoformat()
        stats = {"soft_expired":0, "hot_to_warm":0, "warm_to_cold":0, "hard_deleted":0}

        for part in self._parts.values():
            for id_, idx, meta in part.get_where({"is_active": {"$eq": 1}}):
                if meta["expires_at"] <= now_iso:
                    meta = deepcopy(meta)
                    meta["is_active"] = 0
                    part.metadata[idx] = meta
                    stats["soft_expired"] += 1

        to_move = self.hot.get_where({"is_active": {"$eq": 1},
                                      "created_at": {"$lte": hot_cutoff}})
        ids_to_remove = []
        for id_, _, meta in to_move:
            if "warm" not in CATEGORY_PARTITIONS.get(meta["category"], []):
                ids_to_remove.append(id_); continue
            new_meta = deepcopy(meta)
            new_meta["partition"] = "warm"
            vec = self.hot.matrix[self.hot._id_index[id_]].tolist()
            self.warm.upsert(id_, vec, new_meta)
            ids_to_remove.append(id_)
            stats["hot_to_warm"] += 1
        self.hot.delete(ids_to_remove)

        to_move = self.warm.get_where({"is_active": {"$eq": 1},
                                       "created_at": {"$lte": warm_cutoff}})
        ids_to_remove = []
        for id_, _, meta in to_move:
            if "cold" not in CATEGORY_PARTITIONS.get(meta["category"], []):
                ids_to_remove.append(id_); continue
            new_meta = deepcopy(meta)
            new_meta["partition"] = "cold"
            vec = self.warm.matrix[self.warm._id_index[id_]].tolist()
            self.cold.upsert(id_, vec, new_meta)
            ids_to_remove.append(id_)
            stats["warm_to_cold"] += 1
        self.warm.delete(ids_to_remove)

        old = self.cold.get_where({"created_at": {"$lte": cold_cutoff}})
        if old:
            self.cold.delete([id_ for id_, _, _ in old])
            stats["hard_deleted"] += len(old)

        return stats

    # ── Stats ─────────────────────────────────────────────────────────────

    def stats(self):
        print("\n── Collection Stats ─────────────────────────────────")
        for pname, part in self._parts.items():
            active     = sum(1 for m in part.metadata if m.get("is_active") == 1)
            cat_counts = {}
            for m in part.metadata:
                cat_counts[m["category"]] = cat_counts.get(m["category"], 0) + 1
            print(f"\n  posts_{pname:<5}  total={part.count:>5,}  active={active:>5,}")
            for cat, n in sorted(cat_counts.items(), key=lambda x: -x[1]):
                print(f"    {cat:<20} {n:>5,}")

print("PostVectorStore class defined.")

PostVectorStore class defined.


## Cell 5 — Initialise the store
Instantiates all 3 partitions. If `.pkl` files already exist on disk, they are loaded automatically.

In [24]:
store = PostVectorStore()
print("\nStore ready.")
print(f"  posts_hot  : {store.hot.count:,} vectors")
print(f"  posts_warm : {store.warm.count:,} vectors")
print(f"  posts_cold : {store.cold.count:,} vectors")

  Loaded posts_hot: 2,511 vectors from disk
  Loaded posts_warm: 382 vectors from disk
  Loaded posts_cold: 1,005 vectors from disk

Store ready.
  posts_hot  : 2,511 vectors
  posts_warm : 382 vectors
  posts_cold : 1,005 vectors


## Cell 6 — Load CSV into 3 collections
Re-run this only when loading fresh data. If the store already has data from disk (Cell 5 loaded it), skip this cell.

In [25]:
df    = pd.read_csv(CSV_PATH)
total = len(df)
print(f"Loading {total:,} posts from {CSV_PATH} ...\n")

counts = {"hot": 0, "warm": 0, "cold": 0, "skipped": 0}

for i, (_, row) in enumerate(df.iterrows()):
    result = store.upsert_post(row.to_dict())
    if result:
        counts[result] += 1
    else:
        counts["skipped"] += 1
    if (i + 1) % 2000 == 0:
        print(f"  {i+1:,}/{total:,} processed...")

print(f"\nLoad complete:")
print(f"  posts_hot  : {counts['hot']:>6,}")
print(f"  posts_warm : {counts['warm']:>6,}")
print(f"  posts_cold : {counts['cold']:>6,}")
print(f"  skipped    : {counts['skipped']:>6,}  (category expired for that age bucket)")
store.stats()

Loading 10,000 posts from posts_vector_db.csv ...

  2,000/10,000 processed...
  4,000/10,000 processed...
  6,000/10,000 processed...
  8,000/10,000 processed...
  10,000/10,000 processed...

Load complete:
  posts_hot  :  2,511
  posts_warm :    382
  posts_cold :  1,005
  skipped    :  6,102  (category expired for that age bucket)

── Collection Stats ─────────────────────────────────

  posts_hot    total=2,511  active=2,511
    deal_req             1,036
    market_update          735
    discussion             385
    knowledge              355

  posts_warm   total=  382  active=  382
    deal_req               225
    discussion              87
    knowledge               70

  posts_cold   total=1,005  active=1,005
    knowledge            1,005


## Cell 7 — Run expiry job
Simulated at `CUTOFF` time. In production this runs every hour.

In [26]:
print("Running expiry job at", CUTOFF)
stats = store.run_expiry_job(reference_time=CUTOFF)

print(f"\n  Soft-expired  : {stats['soft_expired']:,}   (is_active flipped to 0)")
print(f"  hot → warm    : {stats['hot_to_warm']:,}   (aged past 72 hrs, still valid)")
print(f"  warm → cold   : {stats['warm_to_cold']:,}   (aged past 5 days, still valid)")
print(f"  Hard-deleted  : {stats['hard_deleted']:,}   (older than 30 days, removed)")

store.stats()

Running expiry job at 2026-04-03 10:00:00+00:00

  Soft-expired  : 236   (is_active flipped to 0)
  hot → warm    : 0   (aged past 72 hrs, still valid)
  warm → cold   : 0   (aged past 5 days, still valid)
  Hard-deleted  : 0   (older than 30 days, removed)

── Collection Stats ─────────────────────────────────

  posts_hot    total=2,511  active=2,275
    deal_req             1,036
    market_update          735
    discussion             385
    knowledge              355

  posts_warm   total=  382  active=  382
    deal_req               225
    discussion              87
    knowledge               70

  posts_cold   total=1,005  active=1,005
    knowledge            1,005


## Cell 8 — Save to disk

In [27]:
store.save_all()

for f in sorted(STORE_DIR.iterdir()):
    size_kb = f.stat().st_size // 1024
    print(f"  {f.name:<25}  {size_kb:>5,} KB")

Saved all partitions to disk.
  posts_cold.pkl               326 KB
  posts_hot.pkl                814 KB
  posts_warm.pkl               124 KB


## Cell 9 — Demo query: rice trader · Mumbai looking for deals

In [ ]:
# USER VECTOR — who this user is
# user_role is a single value (a user has one role)
# no category — users don't have categories
# qty is the user's own typical trade size from their profile

user_vec_deal = PostVectorStore.build_user_vector(
    commodity  = "rice",
    user_role  = "trader",
    latitude   = 19.0760,    # Mumbai
    longitude  = 72.8777,
    qty_min_mt = 50,         # this trader typically deals 50–200 MT
    qty_max_mt = 200,
)



print(f"User vector dims : {len(user_vec_deal)}")
print(f"Commodity dims   : {user_vec_deal[0:10]}   (rice=1 at index 0)")
print(f"Role dims        : {user_vec_deal[10:13]}  (trader=1 at index 0)")
print(f"Geo dims         : {[round(x,4) for x in user_vec_deal[13:16]]}")
print(f"Qty dims         : {user_vec_deal[16:18]}  (normalised 50/5000, 200/5000)")

# Query — filter by category and commodity in metadata, not in vector
results = store.query(
    user_vector = user_vec_deal,
    filters     = {"is_active": 1},
    top_k       = 10,
    seen_ids    = set(),
)

print(f"\nTop {len(results)} deal_req results for rice trader in Mumbai:\n")
print(f"{'#':<3} {'score':<8} {'partition':<8} {'city':<18} "
      f"{'qty_min':>8} {'qty_max':>8} {'price/MT':>10} {'created':<12}")
print("─" * 88)
for i, r in enumerate(results, 1):
    print(f"{i:<3} {r['score']:<8.4f} {r['partition']:<8} "
          f"{r['location_city']:<18} "
          f"{r['category']:<12} "
          f"{int(r['qty_min_mt']):>8,} {int(r['qty_max_mt']):>8,} "
          f"{int(r['price_inr']):>10,} "
          f"{r['created_at'][:10]:<12}")

User vector dims : 18
Commodity dims   : [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]   (rice=1 at index 0)
Role dims        : [1.0, 0.0, 0.0]  (trader=1 at index 0)
Geo dims         : [0.2782, 0.9032, 0.3268]
Qty dims         : [0.01, 0.04]  (normalised 50/5000, 200/5000)

Top 10 deal_req results for rice trader in Mumbai:

#   score    partition city                qty_min  qty_max   price/MT created     
────────────────────────────────────────────────────────────────────────────────────────
1   0.9997   hot      Mumbai             market_update        0        0          0 2026-04-01  
2   0.9997   hot      Pune               market_update        0        0          0 2026-04-02  
3   0.9992   hot      Rajkot             deal_req           93      333     35,701 2026-04-01  
4   0.9991   hot      Rajkot             deal_req           63      341     38,439 2026-03-31  
5   0.9991   hot      Ahmedabad          deal_req          160      302     29,554 2026-03-31  
6   0.9990  

In [51]:
user_taste_profile = {
    "deal_req":      103632,
    "market_update": 735,
    "discussion":    385,
    "knowledge":     120,
}

def rerank_with_taste(candidates: list, taste_profile: dict) -> list:
    total = sum(taste_profile.values())
    
    # normalize to weights (deal_req=0.457, market=0.324, discussion=0.170, knowledge=0.053)
    weights = {cat: count / total for cat, count in taste_profile.items()}
    
    # final score = vector similarity × category taste weight
    for post in candidates:
        category_weight      = weights.get(post["category"], 0.1)
        post["final_score"]  = post["score"] * category_weight
    
    # sort by final score
    candidates.sort(key=lambda x: x["final_score"], reverse=True)
    return candidates

# your existing results list — just pass it in
reranked = rerank_with_taste(results, user_taste_profile)

# take top 25 for feed
feed = reranked[:50]
# print(feed)
for r in feed[:-1]:
    print(f"{r['final_score']:.4f}  {r['category']:<15}  {r['score']:.4f}  {r['location_city']}   ")

0.9873  deal_req         0.9992  Rajkot   
0.9873  deal_req         0.9991  Rajkot   
0.9872  deal_req         0.9991  Ahmedabad   
0.0070  market_update    0.9997  Mumbai   
0.0070  market_update    0.9997  Pune   
0.0070  market_update    0.9990  Rajkot   
0.0070  market_update    0.9989  Ahmedabad   
0.0011  knowledge        0.9990  Rajkot   
0.0011  knowledge        0.9990  Rajkot   


## Cell 10 — Demo query: wheat exporter · Punjab looking for market updates

In [ ]:
user_vec_2 = PostVectorStore.build_vector(
    commodity        = "wheat",
    target_roles_str = "exporter",
    latitude         = 30.9010,    # Ludhiana
    longitude        = 75.8573,
    category         = "discussion",
)

results_2 = store.query(
    user_vector = user_vec_2,
    filters     = {"is_active": 1, "category": "market_update", "commodity": "wheat"},
    top_k       = 10,
    seen_ids    = set(),
)

print(f"{'#':<3} {'score':<8} {'partition':<8} {'city':<18} {'created':<12} {'expires':<12}")
print("─" * 72)
for i, r in enumerate(results_2, 1):
    print(f"{i:<3} {r['score']:<8.4f} {r['partition']:<8} "
          f"{r['location_city']:<18} "
          f"{r['created_at'][:10]:<12} {r['expires_at'][:10]:<12}")
    

#   score    partition city               created      expires     
────────────────────────────────────────────────────────────────────────
1   1.0000   hot      Ludhiana           2026-04-03   2026-04-05  
2   1.0000   hot      Ludhiana           2026-04-03   2026-04-05  
3   1.0000   hot      Chandigarh         2026-04-02   2026-04-04  
4   0.9999   hot      Amritsar           2026-04-02   2026-04-04  
5   0.9998   hot      Hisar              2026-04-02   2026-04-04  
6   0.9998   hot      Hisar              2026-04-01   2026-04-03  
7   0.9996   hot      Gurugram           2026-04-02   2026-04-04  
8   0.9991   hot      Agra               2026-04-02   2026-04-04  
9   0.9986   hot      Jodhpur            2026-04-02   2026-04-04  
10  0.9982   hot      Kanpur             2026-04-03   2026-04-05  


## Cell 11 — Simulate a new post being published
This is what your backend calls the moment a user hits Publish.

In [12]:
import uuid

new_post = {
    "post_id":           str(uuid.uuid4()),
    "category":          "deal_req",
    "commodity":         "rice",
    "target_roles":      "trader|broker",
    "location_state":    "Maharashtra",
    "location_city":     "Mumbai",
    "latitude":          19.0760,
    "longitude":         72.8777,
    "qty_min_mt":        150,
    "qty_max_mt":        400,
    "price_per_unit_inr":32000,
    # created_at = NOW (just published)
    "created_at":        CUTOFF.isoformat(),
    # expires_at = created_at + CATEGORY_EXPIRY_DAYS[category]
    "expires_at":        (CUTOFF + timedelta(days=CATEGORY_EXPIRY_DAYS["deal_req"])).isoformat(),
}

partition = store.upsert_post(new_post)
print(f"New post written to partition: posts_{partition}")
print(f"  post_id    : {new_post['post_id']}")
print(f"  created_at : {new_post['created_at'][:19]}")
print(f"  expires_at : {new_post['expires_at'][:19]}")
print(f"  hot count now: {store.hot.count:,}")

New post written to partition: posts_hot
  post_id    : 7cd40be7-95bd-407d-986c-0561bc155e3c
  created_at : 2026-04-03T10:00:00
  expires_at : 2026-04-10T10:00:00
  hot count now: 2,512
